# One-Class SVM Probabilistic Outputs — Demo (ART1)

Reproduces the four methods from Que & Lin, IEEE TNNLS 2025:
1. **Platt scaling** (baseline — paper argues it collapses to 0/1)
2. **Binning equidistantly**
3. **Binning by density** (LIBSVM ≥ 3.3 default)
4. **New Gamma scaling** (paper's parametric proposal)

Dataset: **ART1** — 1-D Gaussian, 25% outliers, ground-truth probabilities available.

## 1. Setup & download data

In [ ]:
!pip install -q scikit-learn scipy matplotlib numpy

In [ ]:
# Download ART1 from the authors' supplementary archive (mirrored from cjlin's site)
import urllib.request, os
BASE = 'https://www.csie.ntu.edu.tw/~cjlin/papers/oneclass_prob/paper_code.zip'
# Quicker: grab files individually after extracting. For Colab, the simplest is to upload art1/art1.t/art1_prob.t
# or fetch the zip. We fetch the zip once (~154 MB):
if not os.path.exists('paper_code.zip'):
    print('Downloading authors\' supplementary code (~154 MB) ...')
    urllib.request.urlretrieve(BASE, 'paper_code.zip')

import zipfile
with zipfile.ZipFile('paper_code.zip') as z:
    for name in ['paper_code/data/art1', 'paper_code/data/art1.t', 'paper_code/data/art1_prob.t']:
        z.extract(name, '.')
print('Done.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
from scipy.stats import gamma as gamma_dist
from scipy.special import erf

def load_libsvm_1d(path):
    xs = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            # parts[0] is label (all '1' for one-class), parts[1:] are 'idx:val'
            x = float(parts[1].split(':')[1])
            xs.append(x)
    return np.array(xs).reshape(-1, 1)

X_train = load_libsvm_1d('paper_code/data/art1')
X_test  = load_libsvm_1d('paper_code/data/art1.t')
p_true  = np.loadtxt('paper_code/data/art1_prob.t')
print(f'Train: {X_train.shape}, Test: {X_test.shape}, Ground-truth probs: {p_true.shape}')

## 2. Fit One-Class SVM

Following paper §V-C: `ν = 0.25` (25% outliers) and `γ = 0.0001` (small γ ⇒ near-hyperspherical decision boundary, RBF kernel).

In [ ]:
NU, GAMMA = 0.25, 0.0001
clf = OneClassSVM(kernel='rbf', nu=NU, gamma=GAMMA).fit(X_train)

f_train = clf.decision_function(X_train).ravel()
f_test  = clf.decision_function(X_test).ravel()
print(f'f_train  range: [{f_train.min():.4f}, {f_train.max():.4f}]')
print(f'predicted outlier ratio (train): {(f_train < 0).mean():.3f}')

## 3. Four probability methods

All take training decision values `f_train` to fit, then map test `f_test` → `P(normal | x)`.

In [ ]:
# --- 3.1 Platt scaling (with predicted labels, since no true labels). Lin et al. 2007 stable form. ---
def platt_fit(f, y):
    Np = (y == 1).sum(); Nn = (y == -1).sum()
    t = np.where(y == 1, (Np + 1) / (Np + 2), 1.0 / (Nn + 2))
    def nll(ab):
        A, B = ab
        z = A * f + B
        # log(1 + exp(z)) stable
        return np.sum(t * z + np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - z * (z < 0))  # numeric form
    # simpler: directly
    def nll2(ab):
        A, B = ab
        z = A * f + B
        return np.sum(t * z + np.logaddexp(0, -z))
    res = minimize(nll2, x0=[0.0, np.log((Nn + 1) / (Np + 1))], method='Nelder-Mead')
    return res.x

def platt_predict(f, AB):
    A, B = AB
    return 1.0 / (1.0 + np.exp(A * f + B))

y_train_pred = np.where(f_train >= 0, 1, -1)
AB = platt_fit(f_train, y_train_pred)
p_platt = platt_predict(f_test, AB)
print('Platt (A, B):', AB)

In [ ]:
# --- 3.2 Binning equidistantly (paper §IV-A) ---
def binning_equidistant(f_train, f_test, n_bins_side=5, eps=0.001):
    fmin, fmax = f_train.min(), f_train.max()
    neg_marks = np.linspace(fmin, 0, n_bins_side + 1)[:-1]   # fmin .. fmin/5
    pos_marks = np.linspace(0, fmax, n_bins_side + 1)[1:]    # fmax/5 .. fmax
    marks = np.concatenate([neg_marks, [0.0], pos_marks])
    # 11 marks → probs 0.001, 0.1, 0.2, ..., 0.9, 0.999 (clip endpoints per paper)
    probs = np.linspace(0, 1, 2 * n_bins_side + 1)
    probs[0], probs[-1] = eps, 1 - eps
    # assign each test point to nearest mark
    idx = np.argmin(np.abs(f_test[:, None] - marks[None, :]), axis=1)
    return probs[idx]

p_eq = binning_equidistant(f_train, f_test)

In [ ]:
# --- 3.3 Binning by density (paper §IV-A, LIBSVM default) ---
def binning_density(f_train, f_test, n_bins_side=5, eps=0.001):
    neg = np.sort(f_train[f_train < 0])
    pos = np.sort(f_train[f_train >= 0])
    # 5 group centers on each side at quantile midpoints
    def centers(arr, n):
        if len(arr) == 0:
            return np.array([])
        qs = (np.arange(n) + 0.5) / n
        return np.quantile(arr, qs)
    neg_marks = centers(neg, n_bins_side)
    pos_marks = centers(pos, n_bins_side)
    marks = np.concatenate([neg_marks, [0.0], pos_marks])
    probs = np.linspace(0, 1, 2 * n_bins_side + 1)
    probs[0], probs[-1] = eps, 1 - eps
    idx = np.argmin(np.abs(f_test[:, None] - marks[None, :]), axis=1)
    return probs[idx]

p_den = binning_density(f_train, f_test)

In [ ]:
# --- 3.4 New Gamma scaling (paper §IV-B). Verbatim port of authors' exp_mse.py. ---
def new_gamma_scaling(f_train, f_test):
    fmax = f_train.max()
    S_train = np.maximum(fmax - f_train, 0)  # regularized decision values
    mu, var = S_train.mean(), S_train.var()
    k, theta = mu * mu / var, var / mu
    S_test = np.maximum(fmax - f_test, 0)
    cdf = gamma_dist.cdf(S_test, a=k, scale=theta)
    cdf_max = gamma_dist.cdf(fmax, a=k, scale=theta)  # = P(S <= fmax) = prob at f=0
    # In authors' code they compute probs = 1 - cdf, prob_dec0 = 1 - cdf_max,
    # then scale so P(normal | f=0) = 0.5. We follow eq. (40):
    p_normal = 1 - cdf
    p_dec0   = 1 - cdf_max
    return np.where(
        p_normal >= p_dec0,
        0.5 + 0.5 * (p_normal - p_dec0) / (1 - p_dec0),
        0.5 * p_normal / p_dec0,
    )

p_gam = new_gamma_scaling(f_train, f_test)

## 4. Compare — MSE against ground truth

In [ ]:
results = {
    'Platt scaling':         mean_squared_error(p_true, p_platt),
    'Binning equidistant':   mean_squared_error(p_true, p_eq),
    'Binning by density':    mean_squared_error(p_true, p_den),
    'New Gamma scaling':     mean_squared_error(p_true, p_gam),
}
print(f'{"Method":<22} {"MSE":>10}    (Table I, ART1)')
print('-' * 50)
for k, v in results.items():
    print(f'{k:<22} {v:10.6f}')
print('\nPaper Table I ART1 reference:')
print('  Platt  0.077723 | eq 0.026122 | density 0.001056 | new Gamma 0.000003')

## 5. Plot — P(normal | f) vs decision value (replicates Fig. 5a)

In [ ]:
order = np.argsort(f_test)
fs = f_test[order]

plt.figure(figsize=(9, 6))
plt.plot(fs, p_true[order],  'r.', ms=2, label='Ideal',               alpha=0.5)
plt.plot(fs, p_platt[order], 'g.', ms=2, label='Platt scaling',       alpha=0.5)
plt.plot(fs, p_eq[order],    'c.', ms=2, label='Binning equidistant', alpha=0.5)
plt.plot(fs, p_den[order],   'b.', ms=2, label='Binning by density',  alpha=0.5)
plt.plot(fs, p_gam[order],   'k.', ms=2, label='New Gamma scaling',   alpha=0.5)
plt.xlabel('Decision value $f$')
plt.ylabel('$P(\\mathrm{normal}\\,|\\,f)$')
plt.title('ART1: probability vs decision value')
plt.legend(markerscale=4)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Q–Q plot (replicates Fig. 7 for ART1)

In [ ]:
# Theoretical quantiles from rescaled percentiles of training decision values
f_sorted = np.sort(f_train)
def percentile_of(f, sorted_arr):
    return np.searchsorted(sorted_arr, f, side='right') / len(sorted_arr)

prc0 = percentile_of(0.0, f_sorted)
prc_test = percentile_of(f_test, f_sorted)
theo = np.where(prc_test >= prc0,
                0.5 + 0.5 * (prc_test - prc0) / (1 - prc0),
                0.5 * prc_test / prc0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p, title in zip(axes,
                        [p_platt, p_den, p_gam],
                        ['Platt', 'Binning by density', 'New Gamma scaling']):
    ax.plot([0, 1], [0, 1], 'r-', lw=1)
    ax.plot(np.sort(theo), np.sort(p), 'b.', ms=2)
    ax.set_xlabel('Theoretical quantile'); ax.set_ylabel('Sample quantile')
    ax.set_title(title); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Takeaways

- **Platt** collapses to a near-step function — confirms §II-C of the paper.
- **Binning equidistant** gives a coarse staircase, sensitive to outlier-stretched range.
- **Binning by density** tracks the ideal curve closely with simple, kernel-agnostic logic.
- **New Gamma scaling** is the most accurate on ART1 (smallest MSE) thanks to the Gamma fit of `f_max - f(x)`.